![title](imagens/projeto23.jpg)

# Projeto 23 - Previsão de Sobrevivência ao Câncer de Mama
### Usando 5 algoritmos de ML.

O câncer de mama é um dos tipos de câncer que se inicia na mama.

 Ele ocorre em mulheres, mas homens também podem desenvolver câncer de mama.

 É a segunda principal causa de morte em mulheres.

 Como o uso de dados na área da saúde é muito comum hoje em dia, podemos utilizar o aprendizado de máquina para prever se um paciente sobreviverá a uma doença mortal como o câncer de mama ou não.

Esta base de dados apresenta dados reais. Fonte: Kaggle (https://www.kaggle.com/datasets/amandam1/breastcancerdataset/data) vivo ou morto].


### Contexto
Este conjunto de dados consiste em um grupo de pacientes com câncer de mama que passaram por cirurgia para remover o tumor. O conjunto de dados é composto pelas seguintes variáveis:

- **Patient_ID:** identificador único do paciente
- **Age:** idade no momento do diagnóstico (anos)
- **Gender:** Masculino/Feminino
- **Protein1, Protein2, Protein3, Protein4:** níveis de expressão (unidades não definidas)
- **Tumour_Stage:** I, II, III
- **Histology:** Carcinoma Ductal Infiltrante, Carcinoma Lobular Infiltrante, Carcinoma Mucinoso
- **ER status:** Positivo/Negativo
- **PR status:** Positivo/Negativo
- **HER2 status:** Positivo/Negativo
- **Surgery_type:** Lumpectomia, Mastectomia Simples, Mastectomia Radical Modificada, Outro
- **Date_of_Surgery:** Data em que a cirurgia foi realizada (em DD-MON-YY)
- **Date_of_Last_Visit:** Data da última visita (em DD-MON-YY) [pode ser nulo, caso o paciente não tenha retornado após a cirurgia]
- **Patient_Status:** Vivo/Morto [pode ser nulo, caso o paciente não tenha retornado após a cirurgia e não haja informação disponível sobre se o paciente está vivo ou morto].

In [ ]:
# Versão da Linguagem Python
from platform import python_version
print('Versão da Linguagem Python usada neste Projeto no Jupyter Notebook:', python_version())

In [ ]:
# Para atualizar um pacote, execute o comando abaixo no terminal ou prompt de comando:
# pip install -U nome_pacote

# Para instalar a versão exata de um pacote, execute o comando abaixo no terminal ou prompt de comando:
# !pip install nome_pacote==versão_desejada

# Depois de instalar ou atualizar o pacote, reinicie o jupyter notebook.

# Instala o pacote watermark. 
# Esse pacote é usado para gravar as versões de outros pacotes usados neste jupyter notebook.
# !pip install -q -U watermark


### Importação das bibliotecas / pacotes necessários em nosso projeto

In [ ]:
# Os nossos pacotes tradicionais
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

# Os pacotes para normalização e preparação dos dados
import sklearn  #para identificação da versão
from sklearn.preprocessing import LabelEncoder, OrdinalEncoder
from sklearn.model_selection import train_test_split

# Os pacotes destinados a aprendizagem de máquina (algoritmos)
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.model_selection import GridSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier

# Importando o método SMOTE da biblioteca imblearn
from imblearn.over_sampling import SMOTE

# Dentro da biblioteca <'imbalanced-learn'> existe um método <'SMOTE'> que significa: Synthetic Minority Over-sampling Technique. 
# Ou seja... O SMOTE é uma técnica de oversampling que cria novas instâncias sintéticas da classe minoritária no conjunto de dados,
# a fim de balancear a distribuição das classes.

# Os pacotes destinados a report de métricas dos modelos
from sklearn.metrics import roc_curve, roc_auc_score, confusion_matrix, accuracy_score, classification_report

# para evitar mensagens de alerta/warnings.
import warnings
warnings.filterwarnings("ignore")

# Carregar o módulo de funções para limpeza de dados
from limpeza_dados import *

In [ ]:
# Versões dos pacotes usados neste jupyter notebook
%reload_ext watermark
%watermark -a "pyPRO - Seja um Profissional Python!" --iversions

In [ ]:
# Importação do dataset
brca = pd.read_csv("dados/BRCA.csv")

In [ ]:
# Visualização do cabeçalho dos dados
brca.head()

## Limpeza dos Dados Nulos

In [ ]:
# verificando se existe algum valor nulo
calcular_porcentagem_valores_ausentes(brca)

In [ ]:
# Exibindo o relatório de valores nulos por coluna
relatorio_valores_ausentes_por_coluna(brca)

In [ ]:
# Limpando ...
# Limpando os registros que não possuem Status (vivo ou morto)
brca.dropna(subset='Patient_Status', inplace = True)

In [ ]:
# verificando se existe algum valor nulo
calcular_porcentagem_valores_ausentes(brca)

In [ ]:
# Exibindo o relatório de valores nulos por coluna
relatorio_valores_ausentes_por_coluna(brca)

In [ ]:
# limpando esses últims 4 valores nulos
brca.dropna(subset='Date_of_Last_Visit', inplace = True)

In [ ]:
# verificando se existe algum valor nulo
calcular_porcentagem_valores_ausentes(brca)

In [ ]:
# Exibindo o relatório de valores nulos por coluna
relatorio_valores_ausentes_por_coluna(brca)

Ótimo! Não temos mais valores nulos.

## Análise exploratória dos dados (EDA - Exploratory Data Analysis)

In [ ]:
# Vamos obter as informações do dataset
brca.info()

Vamos fazer uma verificação se existem apenas mulheres na base de dados

In [ ]:
brca['Gender'].value_counts()

O Gênero como uma característica no treinamento do modelo provavelmente não será útil, pois não há ocorrências suficientes de pacientes do sexo masculino.

Vamos fazer uma distribuição por idade dos pacientes...

In [ ]:
brca['Age'].hist(bins = 50, grid = False)
plt.xlabel(xlabel = "Age")
plt.ylabel(ylabel = "Count")
plt.title("Age Distribution of BRCA Patients")
plt.show()

Os grupos etários no conjunto de dados parecem estar distribuídos principalmente de forma normal, com leve assimetria à direita - a maioria da população está no grupo etário de 45 a 55 anos.


Vamos verificar agora a classificação única de tipo de câncer de mama nos pacientes

In [ ]:
brca['Histology'].unique()

Agora vamos exisbir um gráfico mostrando a insidência desses tipos de cancer na população do dataset

In [ ]:
def plot_pie_chart(column, title):
    # define Seaborn color palette to use
    palette_color = sns.color_palette('bright')
  
    # plotting data on chart
    plt.pie(brca[column].value_counts(), labels=brca[column].unique(), colors=palette_color, autopct='%.0f%%')

    plt.title(title)
    # displaying chart
    plt.show()

In [ ]:
plot_pie_chart("Histology", "Carincoma Classifications in Patient Dataset")

### Carcinoma Ductal Infiltrante (CDI)

+ O Carcinoma Ductal Infiltrante (CDI), também conhecido como carcinoma ductal invasivo, é um tipo de câncer de mama que começa nos ductos de leite da mama e se move para os tecidos próximos. Com o tempo, o CDI pode se espalhar (metastatizar) pelos gânglios linfáticos ou pela corrente sanguínea para outras áreas do corpo. (Mais informações: https://www.cancer.gov/types/breast/patient/invasive-ductal-carcinoma-treatment-pdq).

### Carcinoma Mucinoso

+ No carcinoma mucinoso, as células cancerígenas se formam no muco, o principal componente do muco. Mucinas são proteínas que ajudam na função das células saudáveis. No carcinoma mucinoso, o muco ao redor das células cancerígenas se torna parte do tumor. O carcinoma mucinoso pode ocorrer em qualquer parte do seu corpo, mas é mais comum na mama. (Mais informações: https://www.cancer.net/cancer-types/mucinous-carcinoma).

### Carcinoma Lobular Infiltrante (CLI)

+ O Carcinoma Lobular Infiltrante, também conhecido como carcinoma lobular invasivo, começa nas glândulas produtoras de leite (lóbulos) da mama. Como um tipo invasivo de câncer, o CLI se espalhou além do local original do tumor. Com o tempo, o CLI pode se tornar um câncer de mama metastático. (Mais informações: https://www.cancer.org/cancer/breast-cancer/understanding-a-breast-cancer-diagnosis/types-of-breast-cancer/invasive-lobular-carcinoma.html).

In [ ]:
# Vamos verificar agora uma distribuição por estágio, sabendo-se que o estágio I é o inicial e o estágio III o mais avançado.
plot_pie_chart("Tumour_Stage", "Tumour Stage Classifications in Patient Dataset")

Vamos agora identificar no dataset a distribuição da variável alvo (Patient Status)

In [ ]:
# Identificando a variável alvo para o modelo de predição (duas categorias: vivo (alive) ou morto (Dead))
plot_pie_chart("Patient_Status", "Patient Status (Target) in Patient Dataset")

Identificamos que o conjunto de dados para nosso alvo está bastante desequilibrado em relação aos pacientes vivos. Isso tornaria difícil fazer classificações adequadas sem reamostrar o conjunto de dados.


## Preprocessamento dos Dados / Engenharia de Atributos

In [ ]:
# Fazendo uma cópia do dataset original e detetando os atributos que não são representativos
brca_processed = brca.copy().drop(columns=['Patient_ID', 'Surgery_type', 'Gender', 'ER status', 'PR status'])

In [ ]:
# Encode da coluna objetivo (target feature) para valor numérico (inteiro) para o processo de classificação
le = LabelEncoder()
brca_processed['Patient_Status_le'], brca_processed['HER2_Status_le'] = le.fit_transform(brca_processed['Patient_Status']), le.fit_transform(brca_processed['HER2 status'])

In [ ]:
# Encode de valores ordinais para o estágio do tumor: categorico para numerico
oe = OrdinalEncoder(dtype=int)
brca_processed['Tumour_Stage_oe'] = oe.fit_transform(np.array(brca_processed['Tumour_Stage']).reshape(-1,1))
brca_processed.drop(columns=['Tumour_Stage', 'Date_of_Surgery', 'Date_of_Last_Visit'], inplace=True)

In [ ]:
# Encode binário da coluna Histology ou seja, o tipo de cancer.
# O método get_dummies() do pandas é usado para criar variáveis dummy a partir de uma variável categórica. 
# Variáveis dummy são variáveis ​​binárias que representam categorias distintas.
brca_processed = pd.get_dummies(brca_processed, columns=['Histology'], dtype = int)

In [ ]:
brca_processed.info()

Como dito, temos 80% de pessoas que sobreviveram e apenas 20% das que morreram.
Isso, no treinamento, causa desequilíbrio, deixando o modelo tendencioso.
Temos que fazer o possível para equalizar os dados.... Existem muitas formas. Uma das mais utilizadas é a criação de dados sintéticos para a classe minoritária.

In [ ]:
# dividindo o dataset
X_train, X_test, y_train, y_test = train_test_split(brca_processed.drop(columns=['Patient_Status', 'Patient_Status_le', 'HER2 status']), brca_processed['Patient_Status_le'], test_size = 0.15, random_state = 42, stratify=brca_processed['Patient_Status_le'])

In [ ]:
# reamostrar o conjunto de dados para que a característica alvo seja distribuida uniformemente
oversample = SMOTE(n_jobs=100, random_state=0)
X_train, y_train = oversample.fit_resample(X_train, y_train)

In [ ]:
# Colocando os dados já reamostrados em um plot
plt.pie(y_train.value_counts(), labels=brca_processed['Patient_Status'].unique(), colors='bright', autopct='%.0f%%')

plt.title("Resampled Patient Status")
# Exibindo o plot (gráfico)
plt.show()

Agora que temos o dataset devidamente balanceado, vamos fazer o treinamento utilizando quatro algoritmos de regressão: Support Vector Classification (SVC), Regressão Logística (Logistic Regression), Random Forest e Árvore de Decisão.

## Support Vector Classification (SVC)

In [ ]:
# instancia o módulo SVC e indica que o modelo deve calcular as probabilidades das previsões ('probability=True')
svc = SVC(probability=True)
# Definição dos parâmetros que serão testados durante o processo de GridSearchCV. 
# Serão testados dois tipos de kernel: linear e rbf
# O parâmetro c identifica o processo de regularização. Dois serão testados: 1 que é o padrão e um mais elevado com 10.
parameters = {'kernel':('linear', 'rbf'), 'C':[1, 10]}
# instanciamento do GrudSearch, utilizando o estimador svc, com os parâmetros definidos acima 
# e n_jobs define a quantidade de processos serão realizados em paralelo (50).
clf = GridSearchCV(svc, parameters, n_jobs=50)
# ajuste do modelo com os dados do treinamento
clf.fit(X_train, y_train)
# GridSearchCV(estimator=SVC(), param_grid={'C': [1, 100], 'kernel': ('linear', 'rbf')})
# o parâmetro "best_estimator_", com base na validação cruzada, seleciona a combinação de parâmetros que produz o melhor desempenho do modelo
clf.best_estimator_

Melhor estimador escolhido durante o ajuste/GridSearch mostra hiperparâmetros para o modelo.


## Regressão Logística (Logistic Regression)

In [ ]:
# instancia o Modelo de Regressão Logistica com os parâmetros: max_iter: número máximo de iterações.
# Tipo de penalidade utilizada: "elasticnet" --> combina penalidade L1 (lasso) e L2 (Ridge).
# solver = 'saga' --> algoritmo de otimização utlizado. Adequado para elasticnet e funciona bem com grandes volumes de dados esparsos.
# l1_ratio = 1 --> especifica que na mistura entre L1 e L2 no cálculo da penalidade, terá prioridade o L1.
logit = LogisticRegression(max_iter=3500, penalty='elasticnet', solver = 'saga', l1_ratio=1)
# ajuste do modelo aos dados de treinamento
logit.fit(X_train, y_train)
# extraimos as importâncias dos coeficientes ajustados do modelo. Trata-se de um vetor pois estamos lidando com uma classificação binária.
logit_importances = logit.coef_[0]
# criamos uma série para armazenar os coeficientes
logit_feature_importances = pd.Series(index = X_train.columns, data= np.abs(logit_importances))
# ordenamos os coeficientes em orde crescente
logit_feature_importances = logit_feature_importances.sort_values(ascending=True)
# criamos aqui um gráfico de barras horizontais mostrando as importâncias das características
logit_feature_importances.plot(kind='barh', title = 'Logisitic Regression Feature Importances', xlabel = 'Importance', ylabel = 'Features')
plt.show()

Após o treinamento do modelo de Regressão Logística no conjunto de dados, as características mais importantes que contribuem para a classificação do Status do Paciente incluem Histologia de ILC, carcinomas IDC e o paciente ser HER2+/-. Características que não parecem contribuir para a previsão do modelo incluem idade e estágio do tumor. Surpreendentemente, o estágio de um tumor não parece contribuir muito informativamente para a previsão do modelo, apesar do conhecimento de domínio desta característica. Isso pode ser devido à assimetria dos dados em direção ao câncer de estágio III, já que isso é prevalente em ~60% dos pacientes no conjunto de dados. O status HER2 também é uma característica informativa para o modelo, como entendemos pelo HER2++ normalmente como um preditor de casos de câncer.


## Random Forest

In [ ]:
# Inicializa o classificaro RandomForest, utilizando um total de 500 árvores na floresta
forest = RandomForestClassifier(n_estimators=500)
# Ajusta o modelo aos dados de treinamento
forest.fit(X_train, y_train)
# cria um dataframe com a extração das importâncias das características
forest_feature_importance = pd.DataFrame(index = forest.feature_names_in_, data=forest.feature_importances_, columns = ['Feature_Importance'])
# exibe um heatmap com a importância das características
sns.heatmap(forest_feature_importance, annot=True).set(title = "Random Classifier Feature Importance Heatmap", ylabel='Features')
plt.show()

## Árvore de Decisão (Decision Tree)

In [ ]:
# Inicializa o modulo de Árvore de Decisão, indicando que o critério de decisão levará em conta o índice gini como medida de pureza
# e terá uma profundidade máxima de 15 nós.
tree_gini = DecisionTreeClassifier(criterion='gini',max_depth=15)
# Faz o ajuste do modelo aos dados de treinamento
tree_gini.fit(X_train, y_train)
# extrai as importância das características em uma Série
tree_gini_feature_importances = pd.Series(tree_gini.feature_importances_, index=tree_gini.feature_names_in_)
# Exibe um gráfico de barras mostrando as importâncias das características.
sns.barplot(x = tree_gini.feature_names_in_, y = tree_gini.feature_importances_).set(title = "Decision Tree Feature Importance", xlabel='Features', ylabel='Importance')
plt.xticks(rotation = 90) 
plt.show()

Ambos os classificadores de árvore de decisão (Árvore de Decisão e Random Forest) produzem rankings semelhantes para a importância das características nos modelos. Os níveis de expressão da Proteína 2 parecem ser um fator determinante para a classificação do status do paciente, com as outras proteínas também contribuindo para as previsões do modelo. Ao contrário do que foi encontrado anteriormente em nosso treinamento de Regressão Logística, a idade também contribui para as previsões do modelo - isso corresponde à noção preconcebida de que a idade também pode ser um determinante do resultado do paciente com a doença. Curiosamente, as características com menor pontuação de importância são a presença de carcinomas específicos (ILC, IDC, MC), o status HER2 e o estágio do tumor.


Vamos verificar o processo de aprendizagem plotando a curva ROC desses quatro algoritmos/métodos...

### Curva ROC
A curva ROC (Receiver Operating Characteristic) é uma ferramenta comumente utilizada na avaliação de modelos de classificação em aprendizado de máquina. Ela representa graficamente a relação entre a taxa de verdadeiros positivos (TPR) e a taxa de falsos positivos (FPR) para diferentes limiares de classificação.

A TPR, também conhecida como sensibilidade, é a proporção de exemplos positivos que foram corretamente classificados como positivos pelo modelo. Por outro lado, a FPR é a proporção de exemplos negativos que foram erroneamente classificados como positivos pelo modelo.

A curva ROC é plotada com a TPR no eixo y (vertical) e a FPR no eixo x (horizontal). Um modelo ideal teria uma curva ROC que se estendesse verticalmente até o canto superior esquerdo do gráfico, indicando uma alta sensibilidade (TPR) e uma baixa taxa de falsos positivos (FPR) em todos os limiares de classificação. Portanto, quanto mais próxima a curva ROC de um modelo estiver desse canto superior esquerdo, melhor será sua capacidade de distinguir entre as clarfeito.


In [ ]:
# Função para plotar a curva ROC
def plot_ROC(model, title, X, Y): 
    # ROC curve
    fpr, tpr, thresholds = roc_curve(y_true=Y, y_score=model.predict_proba(X)[:, 1])
    roc_auc = roc_auc_score(y_true=Y, y_score=model.predict_proba(X)[:, 1])
    plt.plot(fpr, tpr, color="blue", label="AUC = %0.3f" % roc_auc)
    plt.plot([0, 1], [0, 1], color="red", linestyle="--", lw=1)
    plt.title(title)
    plt.xlabel("False Positive Rate")
    plt.ylabel("True Positive Rate")
    plt.legend(loc="lower right")
    plt.show()

### Plotando as curvas ROC com base nos dados de treinamento

In [ ]:
# Plotando a curva ROC para Regressão Logistica
plot_ROC(logit, "Logisitic Regression ROC Curve", X_train, y_train)
print(f"Accuracy Score: {accuracy_score(y_true=y_train, y_pred=logit.predict(X_train))}")
print(f"Confusion Matrix:\n {confusion_matrix(y_true=y_train, y_pred=logit.predict(X_train))}")
print(f"Classification Report:\n {classification_report(y_true=y_train, y_pred=logit.predict(X_train))}")

In [ ]:
# Plotando a curva ROC para SVC/SVM
plot_ROC(clf, "Support Vector Machine Classifier ROC Curve", X_train, y_train)
print(f"Accuracy Score: {accuracy_score(y_true=y_train, y_pred=clf.predict(X_train))}")
print(f"Confusion Matrix:\n {confusion_matrix(y_true=y_train, y_pred=clf.predict(X_train))}")
print(f"Classification Report:\n {classification_report(y_true=y_train, y_pred=clf.predict(X_train))}")

In [ ]:
# Plotando a curva ROC para Random Forest
plot_ROC(forest, "Random Forest Classifier ROC Curve", X_train, y_train)
print(f"Accuracy Score: {accuracy_score(y_true=y_train, y_pred=forest.predict(X_train))}")
print(f"Confusion Matrix:\n {confusion_matrix(y_true=y_train, y_pred=forest.predict(X_train))}")
print(f"Classification Report:\n {classification_report(y_true=y_train, y_pred=forest.predict(X_train))}")

In [ ]:
# Plotando a curva ROC para Árvore de Decisão
plot_ROC(tree_gini, "Decision Tree Classifier ROC Curve", X_train, y_train)
print(f"Accuracy Score: {accuracy_score(y_true=y_train, y_pred=tree_gini.predict(X_train))}")
print(f"Confusion Matrix:\n {confusion_matrix(y_true=y_train, y_pred=tree_gini.predict(X_train))}")
print(f"Classification Report:\n {classification_report(y_true=y_train, y_pred=tree_gini.predict(X_train))}")

### Vamos agora selecionar apenas os dois melhores algoritmos classificadores e plotar a curva ROC com os dados de Teste

In [ ]:
# Curva ROC Random Forest com dados de Teste
plot_ROC(forest, "Random Forest Classifier ROC Curve", X_test, y_test)
print(f"Accuracy Score: {accuracy_score(y_true=y_test, y_pred=forest.predict(X_test))}")
print(f"Confusion Matrix:\n {confusion_matrix(y_true=y_test, y_pred=forest.predict(X_test))}")
print(f"Classification Report:\n {classification_report(y_true=y_test, y_pred=forest.predict(X_test))}")

In [ ]:
# Curva ROC da Arvore de Decisão dos dados de Teste
plot_ROC(tree_gini, "Decision Tree Classifier ROC Curve", X_test, y_test)
print(f"Accuracy Score: {accuracy_score(y_true=y_test, y_pred=tree_gini.predict(X_test))}")
print(f"Confusion Matrix:\n {confusion_matrix(y_true=y_test, y_pred=tree_gini.predict(X_test))}")
print(f"Classification Report:\n {classification_report(y_true=y_test, y_pred=tree_gini.predict(X_test))}")

### Leitura da Matriz de Confusão acima:
A matriz de confusão é uma tabela que mostra o desempenho de um modelo de classificação em relação aos rótulos reais dos dados. Cada linha representa as instâncias observadas em uma classe real, enquanto cada coluna representa as instâncias previstas em uma classe pelo modelo.

Para a matriz de confusão fornecida:

- Na primeira linha, temos os casos verdadeiros da classe negativa (por exemplo, a classe 0 ou "Negativo"). Dos 39 casos verdadeiros da classe negativa, o modelo classificou corretamente 31 deles como negativos (verdadeiros negativos - TN), mas erroneamente classificou 8 como positivos (falsos positivos - FP).

- Na segunda linha, temos os casos verdadeiros da classe positiva (por exemplo, a classe 1 ou "Positivo"). Dos 9 casos verdadeiros da classe positiva, o modelo classificou corretamente 4 deles como positivos (verdadeiros positivos - TP), mas erroneamente classificou 5 como negativos (falsos negativos - FN).

Portanto, para interpretar esses resultados:

- O modelo classificou corretamente 31 instâncias como negativas e 4 instâncias como positivas.
- O modelo classificou incorretamente 8 instâncias como positivas e 5 instâncias como negativas.

Esses valores podem ser usados para calcular métricas de avaliação do modelo, como precisão, recall, F1-score e outras.


## Avaliação do Modelo

No total, quatro algoritmos de modelo diferentes foram treinados no conjunto de dados de câncer de mamaSVC/SVMte, Regressão LogísticaRandom Forestiae Árvore de Decisão) para classificação binária do status do paciente alvo. Acima, exibimos os gráficos da Curva Característica de Operação do Receptor (Receiver Operating Characteristic - ROC) do desempenho dos modelos (Área Sob a Curva, AUC) com os conjuntos de dados de treinamento/teste para prever o status do paciente com pontuação de precisão, a matriz de confusão e o relatório de classificação.

Os algoritmos de modelo SVC e Regressão Logística parecem estar subajustados durante o treinamento. Retestando nossas previsões do modelo no mesmo conjunto de dados usado para treinamento, observamos erros nas previsões com vários falsos negativos/positivos. O desempenho fraco nos dados de treinamento pode ser porque os modelos são muito simples, o que poderia ser necessário ajustar os hiperparâmetros do modelo (regularização, n_jobs, adição de novas características específicas do domínio) para melhorar o desempenho.

Os classificadorRandom Forestatória e Árvore de Decisão ambos têm um desempenho muito bom nos dados de treinamento, com 100% e 98% de precisão das classes alvo. Com esse resultado, optamos por avançar na avaliação desses modelos para a previsão dos dados de teste.

As pontuaçõesRandom Forestleat75ria ~0.6 (mostradas em azul acima) e dos classificadores de Árvore de Decisão73ão ~0.55. Os modelos realmente dão apenas um leve benefício ao prever o status de um paciente em comparação com adivinhar aleatoriamente o resultado do paciente (equivalente a uma pontuação A7C de 0.5). Além disso, as pontuações de precisão para esses modelos variam em torno de ~70%. Ao analisar as matrizes de confusão/relatório de classificação dos modelos, vemos que para todos eles há uma taxa de falsos positivos bastante alta, especificamente para classificar o paciente como falecido pela doença (classe 1). A precisão, o recall e os escores F1 para cada um dos modelos são bastante baixos para a previsão dessa classe.


### Mesmo não sendo viável, podemos utilizar o modelo, por exemplo do Random Forest para fazer uma previsão...

In [ ]:
# Previsão
# features = [['Age', 'Protein1',	'Protein2',	'Protein3',	'Protein4',	'HER2_Status_le', 'Tumour_Stage_oe', 'Histology_Infiltrating Ductal Carcinoma', 'Histology_Infiltrating', 'Lobular Carcinoma', 'Histology_Mucinous Carcinom']]
features = np.array([[37.0, 0.080353, 0.42638, 0.54715, 0.273680, 1, 3, 1, 0, 0]])
print(forest.predict(features))

## Fim do Projeto 23